<a href="https://colab.research.google.com/github/BlackBoxJU50/Civic_Sense_Classification_Machine_Learning_project/blob/main/Civic_Sense_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()   # Upload your dataset.csv


Saving dataset.csv to dataset.csv


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   MULTIMODAL CIVIC INTELLIGENCE (MCI) — GOOGLE COLAB VERSION    ║
║   Jahangirnagar University — Dept. of CSE                       ║
║                                                                  ║
║   INSTRUCTIONS TO RUN IN GOOGLE COLAB:                          ║
║   1. Open https://colab.research.google.com                     ║
║   2. Click: Runtime > Change runtime type > GPU (T4 free)       ║
║   3. Upload your dataset.csv to Colab or mount Google Drive      ║
║   4. Run each cell from top to bottom                           ║
╚══════════════════════════════════════════════════════════════════╝
"""

# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# Run this first. Runtime will restart automatically.
# ============================================================
# !pip install transformers torch scikit-learn seaborn matplotlib pandas -q


# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE (OPTIONAL)
# Use this if your dataset is stored in Google Drive.
# Otherwise skip to CELL 3 and upload manually.
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/dataset.csv"   # ← adjust path


# ============================================================
# CELL 3 — UPLOAD DATASET FROM LOCAL MACHINE
# Uncomment and run if you want to upload from your computer.
# ============================================================
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = "dataset.csv"   # name of the file you uploaded


# ============================================================
# CELL 4 — IMPORTS & CONFIG
# ============================================================

import os, json, time, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ─── CONFIGURATION ────────────────────────────────────────────
DATA_PATH    = "/content/dataset.csv"      # ← change if needed
RESULTS_DIR  = "/content/experiment_results"
TRAIN_DIR    = "/content/train"
TEST_DIR     = "/content/test"

CFG = {
    "model_name"  : "bert-base-multilingual-cased",
    "max_len"     : 128,
    "batch_size"  : 32,      # Colab GPU handles larger batches
    "epochs"      : 5,       # More epochs for better convergence
    "lr"          : 2e-5,
    "test_size"   : 0.20,
    "seed"        : 42,
    "languages"   : ["English", "Bangla", "Banglish"],
}

LABEL_MAP    = {-1: 2, 0: 0, 1: 1}
ID2LABEL     = {0: "Neutral", 1: "Positive", 2: "Negative"}
TARGET_NAMES = ["Neutral", "Positive", "Negative"]

# Device — automatically picks GPU on Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {device}")
print(f"   GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


# ============================================================
# CELL 5 — HELPER FUNCTIONS
# ============================================================

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_dirs():
    for d in [RESULTS_DIR, TRAIN_DIR, TEST_DIR]:
        os.makedirs(d, exist_ok=True)
    for lang in CFG["languages"]:
        os.makedirs(os.path.join(RESULTS_DIR, lang), exist_ok=True)

seed_everything(CFG["seed"])
make_dirs()
print("✅ Directories created.")


# ============================================================
# CELL 6 — DATASET CLASS
# ============================================================

class CivicDataset(Dataset):
    """
    Tokenises civic narratives with mBERT tokeniser.
    Each item includes input_ids, attention_mask, label,
    and metadata (action, category, feedback, advice, language).
    """
    def __init__(self, records: pd.DataFrame, tokenizer, max_len: int):
        self.records   = records.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row   = self.records.iloc[idx]
        text  = str(row["Action"])

        enc = self.tokenizer(
            text,
            add_special_tokens    = True,
            max_length            = self.max_len,
            padding               = "max_length",
            truncation            = True,
            return_attention_mask = True,
            return_tensors        = "pt",
        )
        return {
            "input_ids"      : enc["input_ids"].flatten(),
            "attention_mask" : enc["attention_mask"].flatten(),
            "label"          : torch.tensor(int(row["Label_Mapped"]), dtype=torch.long),
            "action"         : text,
            "category"       : str(row.get("Category", "")),
            "feedback"       : str(row.get("Feedback", "")),
            "advice"         : str(row.get("Advice",   "")),
            "language"       : str(row.get("Language", "")),
        }

print("✅ CivicDataset class ready.")


# ============================================================
# CELL 7 — DATA LOADING & PARTITIONING
# ============================================================

print("\n📂 Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"   Total rows : {len(df)}")
print(f"   Columns    : {list(df.columns)}")

# Map labels: -1→Negative(2), 0→Neutral(0), 1→Positive(1)
df["Label_Mapped"] = df["Civic_Sense_Label"].map(LABEL_MAP)
df = df.dropna(subset=["Label_Mapped"])

print(f"\n   Label distribution:")
print(df["Civic_Sense_Label"].value_counts().to_string())

# Stratified split — keeps class balance equal in both sets
df_train, df_test = train_test_split(
    df,
    test_size    = CFG["test_size"],
    random_state = CFG["seed"],
    stratify     = df["Label_Mapped"],
)

df_train.to_csv(os.path.join(TRAIN_DIR, "train.csv"), index=False)
df_test .to_csv(os.path.join(TEST_DIR,  "test.csv"),  index=False)

print(f"\n✅ Split complete:")
print(f"   Train rows : {len(df_train)}")
print(f"   Test  rows : {len(df_test)}")

print(f"\n   Language distribution in test set:")
print(df_test["Language"].value_counts().to_string())


# ============================================================
# CELL 8 — MODEL & TOKENIZER SETUP
# ============================================================

print("\n⏳ Loading mBERT tokenizer and model...")
tokenizer = BertTokenizer.from_pretrained(CFG["model_name"])
model     = BertForSequenceClassification.from_pretrained(
    CFG["model_name"], num_labels=3
).to(device)

# ── Class-Weighted Loss (handles imbalanced dataset) ──────────
counts      = df_train["Label_Mapped"].value_counts().sort_index()
total       = counts.sum()
weights_val = [total / (len(counts) * counts[i]) for i in range(len(counts))]
class_weights = torch.tensor(weights_val, dtype=torch.float).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

print(f"✅ Model loaded on {device}")
print(f"   Class weights → {dict(zip(TARGET_NAMES, [round(w,3) for w in weights_val]))}")

# ── Data Loaders ──────────────────────────────────────────────
train_ds = CivicDataset(df_train, tokenizer, CFG["max_len"])
test_ds  = CivicDataset(df_test,  tokenizer, CFG["max_len"])

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=0, pin_memory=False)

# ── Optimizer + Cosine LR Scheduler ──────────────────────────
total_steps = len(train_loader) * CFG["epochs"]
optimizer   = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
scheduler   = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-7)

print(f"\n✅ Ready to train:")
print(f"   Batches/epoch : {len(train_loader)}")
print(f"   Total steps   : {total_steps}")


# ============================================================
# CELL 9 — TRAINING LOOP
# ============================================================

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in loader:
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        out  = model(ids, attention_mask=mask)
        loss = criterion(out.logits, labels)
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = out.logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


print("🚀 Starting full training...\n")
train_losses, train_accs = [], []

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    loss, acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    elapsed   = time.time() - t0
    train_losses.append(loss)
    train_accs.append(acc * 100)
    print(f"  Epoch {epoch}/{CFG['epochs']} | Loss: {loss:.4f} | Acc: {acc*100:.2f}% | {elapsed:.1f}s")

print("\n✅ Training complete!")


# ============================================================
# CELL 10 — EVALUATION
# ============================================================

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            labels = batch["label"]

            out    = model(ids, attention_mask=mask)
            probs  = torch.softmax(out.logits, dim=1).cpu().numpy()
            preds  = out.logits.argmax(dim=1).cpu().tolist()

            y_true.extend(labels.tolist())
            y_pred.extend(preds)
            y_prob.extend(probs.tolist())
    return y_true, y_pred, np.array(y_prob)


print("📊 Evaluating on full test set...")
y_true, y_pred, y_prob = evaluate(model, test_loader, device)

overall_acc = accuracy_score(y_true, y_pred)
macro_f1    = f1_score(y_true, y_pred, average="macro", zero_division=0)
y_bin       = label_binarize(y_true, classes=[0, 1, 2])
try:
    auc = roc_auc_score(y_bin, y_prob, multi_class="ovr", average="macro")
except Exception:
    auc = float("nan")

print(f"\n  ✅ Overall Accuracy : {overall_acc * 100:.2f}%")
print(f"  ✅ Macro F1-Score   : {macro_f1:.4f}")
print(f"  ✅ Macro AUC        : {auc:.4f}")
print("\n  Per-Class Report:")
print(classification_report(y_true, y_pred, target_names=TARGET_NAMES, zero_division=0))


# ============================================================
# CELL 11 — VISUALIZATIONS
# ============================================================

# ── Helper A: Confusion Matrix ────────────────────────────────
def save_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="YlOrRd",
        xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES,
        linewidths=0.5, ax=ax, annot_kws={"size": 14},
    )
    ax.set_xlabel("Predicted Label", fontsize=13)
    ax.set_ylabel("Actual Label",    fontsize=13)
    ax.set_title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

# ── Helper B: Precision / Recall / F1 Grouped Bar Chart ──────
def save_prf_chart(y_true, y_pred, title, save_path):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0
    )
    x       = np.arange(len(TARGET_NAMES))
    width   = 0.25
    colors  = ["#457b9d", "#2a9d8f", "#e76f51"]

    fig, ax = plt.subplots(figsize=(10, 6))
    for idx, (vals, lbl) in enumerate(zip([prec, rec, f1], ["Precision", "Recall", "F1-Score"])):
        bars = ax.bar(x + (idx - 1) * width, vals, width, label=lbl, color=colors[idx])
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                    f"{h:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(TARGET_NAMES, fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

# ── Helper C: Training Loss Curve ────────────────────────────
def save_loss_curve(losses, accs, epochs, save_path):
    fig, ax1 = plt.subplots(figsize=(9, 5))
    color_loss = "#e63946"
    color_acc  = "#457b9d"

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=color_loss)
    ax1.plot(range(1, epochs + 1), losses, marker="o",
             color=color_loss, linewidth=2, label="Loss")
    ax1.tick_params(axis="y", labelcolor=color_loss)

    ax2 = ax1.twinx()
    ax2.set_ylabel("Accuracy (%)", color=color_acc)
    ax2.plot(range(1, epochs + 1), accs, marker="s",
             color=color_acc, linewidth=2, linestyle="--", label="Accuracy")
    ax2.tick_params(axis="y", labelcolor=color_acc)

    plt.title("Training Loss & Accuracy Curve", fontsize=14, fontweight="bold")
    fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
    plt.xticks(range(1, epochs + 1))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

# ── Draw All Overall Visualizations ──────────────────────────
print("📈 Generating visualizations...\n")

save_loss_curve(train_losses, train_accs, CFG["epochs"],
                os.path.join(RESULTS_DIR, "training_loss_curve.png"))

save_confusion_matrix(y_true, y_pred,
    title     = "MCI Full Dataset — Civic Sense Confusion Matrix",
    save_path = os.path.join(RESULTS_DIR, "confusion_matrix_overall.png"))

save_prf_chart(y_true, y_pred,
    title     = "MCI Full Dataset — Precision / Recall / F1 per Class",
    save_path = os.path.join(RESULTS_DIR, "prf_chart_overall.png"))


# ============================================================
# CELL 12 — PER-LANGUAGE EVALUATION
# ============================================================

print("\n🌐 Per-Language Evaluation\n" + "=" * 50)

df_test_pred = df_test.copy().reset_index(drop=True)
df_test_pred["y_pred"] = y_pred
df_test_pred["y_true"] = y_true

for lang in CFG["languages"]:
    lang_dir = os.path.join(RESULTS_DIR, lang)
    df_lang  = df_test_pred[df_test_pred["Language"] == lang]

    if df_lang.empty:
        print(f"\n  [{lang}] — No test samples found, skipping.")
        continue

    yt = df_lang["y_true"].tolist()
    yp = df_lang["y_pred"].tolist()

    lang_acc = accuracy_score(yt, yp)
    lang_f1  = f1_score(yt, yp, average="macro", zero_division=0)

    print(f"\n{'─'*50}")
    print(f"  [{lang}] — {len(df_lang)} test samples")
    print(f"    Accuracy : {lang_acc * 100:.2f}%")
    print(f"    F1 Score : {lang_f1:.4f}")
    print(classification_report(yt, yp, target_names=TARGET_NAMES, zero_division=0))

    # Confusion Matrix
    save_confusion_matrix(
        yt, yp,
        title     = f"MCI — Confusion Matrix [{lang}]",
        save_path = os.path.join(lang_dir, f"confusion_matrix_{lang.lower()}.png"),
    )

    # Precision / Recall / F1 Chart
    save_prf_chart(
        yt, yp,
        title     = f"MCI — Precision / Recall / F1 [{lang}]",
        save_path = os.path.join(lang_dir, f"prf_chart_{lang.lower()}.png"),
    )

    # Save CSV classification report
    report_dict = classification_report(
        yt, yp, target_names=TARGET_NAMES, output_dict=True, zero_division=0
    )
    pd.DataFrame(report_dict).transpose().to_csv(
        os.path.join(lang_dir, f"classification_report_{lang.lower()}.csv")
    )

    # Save JSON summary
    with open(os.path.join(lang_dir, f"summary_{lang.lower()}.json"), "w") as f:
        json.dump({
            "language"        : lang,
            "num_test_samples": len(df_lang),
            "accuracy"        : round(lang_acc, 4),
            "macro_f1"        : round(lang_f1, 4),
        }, f, indent=2)

    # Sample Story Output
    sample  = df_lang.iloc[0]
    pred_id = int(sample["y_pred"])
    story_out = "\n".join([
        "=" * 60,
        "tell me a story :",
        f"  input the story          : {sample['Action']}",
        f"  civic sense is           : {ID2LABEL[pred_id]}",
        f"  catagory of Social Norm  : {sample.get('Category', 'N/A')}",
        f"  Consequences             : {sample.get('Feedback', 'N/A')}",
        f"  Advice                   : {sample.get('Advice', 'N/A')}",
        "=" * 60,
    ])
    print(f"\n  Sample [{lang}]:\n{story_out}")
    with open(os.path.join(lang_dir, f"sample_story_{lang.lower()}.txt"), "w", encoding="utf-8") as f:
        f.write(story_out + "\n")


# ============================================================
# CELL 13 — SAVE SUMMARY METRICS & DOWNLOAD RESULTS
# ============================================================

metrics = {
    "overall_accuracy": round(overall_acc, 4),
    "macro_f1"        : round(macro_f1, 4),
    "macro_auc"       : round(auc, 4) if not np.isnan(auc) else "N/A",
    "epochs_trained"  : CFG["epochs"],
    "batch_size"      : CFG["batch_size"],
    "train_rows"      : len(df_train),
    "test_rows"       : len(df_test),
}
with open(os.path.join(RESULTS_DIR, "summary_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "=" * 60)
print(" ✅ EXPERIMENT COMPLETE")
print(f"    Overall Accuracy : {overall_acc * 100:.2f}%")
print(f"    Macro F1         : {macro_f1:.4f}")
print(f"    AUC              : {auc:.4f}")
print(f"\n    Results folder   : {RESULTS_DIR}")
print("=" * 60)

# Download all results as a ZIP (uncomment in Colab)
# import shutil
# shutil.make_archive("/content/mci_results", "zip", RESULTS_DIR)
# from google.colab import files
# files.download("/content/mci_results.zip")


✅ Device : cpu
   GPU    : N/A
✅ Directories created.
✅ CivicDataset class ready.

📂 Loading dataset...
   Total rows : 7095
   Columns    : ['Category', 'Action', 'Civic_Sense_Label', 'Feedback', 'Advice', 'Language']

   Label distribution:
Civic_Sense_Label
 0    3135
-1    2880
 1    1080

✅ Split complete:
   Train rows : 5676
   Test  rows : 1419

   Language distribution in test set:
Language
Bangla      502
English     474
Banglish    443

⏳ Loading mBERT tokenizer and model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded on cpu
   Class weights → {'Neutral': np.float64(0.754), 'Positive': np.float64(2.19), 'Negative': np.float64(0.821)}

✅ Ready to train:
   Batches/epoch : 178
   Total steps   : 890
🚀 Starting full training...

